# Nearest neighbors
There is a category of ML methods that stores every training sample and bases future inferences on the distance of the sample compared to its nearest training neighbors. These approach is called "nearest neighbors", works on supervised learning in classification and regression tasks, this category is a type of non-generalizing learning as we are just storing our training samples and at inference we compute the distance between a new sample and our training sample following a certain criterion.

## Approaches
### K-Nearest neighbors (KNN)
KNN is the most popular approach, the outputs are computed choosing the $k$ nearest neighbors to the sample. The optimal choice of $k$ is highly data-dependent, small values add noise to the model but have a decision boundary evident; in contrast higher values reduce the noise but the decision boundary is poorly defined. A good strategy for tunning $k$ hyperparameter is using cross-validation.

### Radius nearest neighbors (RNN)
Another approach establishes a radius $r$ where the samples with lower distance than $r$ are considered "close" to a certain sample. It can be weigh up more robust than KNN and actually is because mitigates the problem of having a lot of training samples close to the testing instance, specially when adding distance to the sample to giving more weight to closest instances.

### Nearest centroid classifier
It's a simple algorithm that represents each class by the centroid of its members, behaves similar to K-Means. On non-convex or high variance classes it suffers a performance downgrade; only works for classification tasks.

## Distance calculation algorithms
### Brute force
The most naive neighbor search implementation computes the distances between all pairs of points in the dataset, for $n$ samples and $d$ features, the computational complexity is $O(dn^2)$. Performs well on small datasets but as the number of samples grows it quickly become infesiable.

### K-Dimensional tree (K-D tree)
It's a bit different to the brute force method, the idea is that if point $A$ is very distant from point $B$ and point $B$ is very close to point $C$ then we know that points $A$ and $C$ are very distant without nedding to calculate their distance. A K-D tree is a tree-based data structure for organizing points in a K-dimensional space, acts similar to a binary search tree with each node representing a hyperplane for some feature in the multi dimensional space. This approach builds a balanced binary search tree, so it's too efficient at searches having a computational complexity of $O(log(n))$ for a single sample, overall the computational cost of is $O(dn log(n))$

### Ball tree
Ball tree improvement of KD tree with high dimensional data, the underlying data structure is the same but instead of partition data along cartesian axes as KD tree does, ball tree partitions data in a series of nesting hyper-spheres. Internal nodes are hyperspheres with radius $r$ and centroid $C$, this approach is more expensive at tree construction but for distance calculation just need to compute the distance from the sample and the centroid.

## Distance metrics
There isn't a single way to calculate samples pairwise distance and considering that this is the core of this estimator category, metric selection is as important as choosing optimal values for $r$ or $k$ hyperparameters in RNN and KNN approaches. Let's see the most common metrics:

- **Euclidean:** The closest path between 2 points into an euclidean space.
$$
    d(\vec{x}, \vec{y}) = \sqrt{\sum_{i=1}^{d} (x_i - y_i)^2}
$$
- **Manhattan:** A distance measure for grid spaces.
$$
    d(\vec{x}, \vec{y}) = \sum_{i=1}^{d} |x_i - y_i|
$$
- **Chebyshev:** An approach to compute distances in a grid space but focusing on finding the maximal shift on any dimension.
$$
    d(\vec{x}, \vec{y}) = \operatorname*{max}_{i \in \{1,\ldots, d \}} |x_i - y_i|
$$
- **Minkowksi:** A versatile distance metric that behaves as a generalization of other metrics, introduces a $p$ parameter to allow adaptation to different data shapes.
$$
    d(\vec{x}, \vec{y}) = \sqrt[p]{(\sum_{i=1}^{d} |x_i - y_i|^{p})}
$$

## Implementation
Scikit library implements the following classes:
 - **KNN:** `KNeighborsClassifier` and `KNeighborsRegressor`.
 - **Radius neighbor:** `RadiusNeighborsClassifier` and `RadiusNeighborsRegressor`
 - **Nearest centroid:** `NearestCentroid`.

### Classification
Having $n$ training samples and $c$ classes, at inference it assigns the most frequent class label from the nearest neighbors to the evaluated sample. In some cases is useful to add weights $w_i$ to every neighbor based on how close is to the sample, however the most common approach sets $w_i = 1$ giving the same treatment to every training sample.

$$
    \hat{y}(x) = \operatorname*{arg\,max}_{c \in \{1,\ldots, C \}} \sum_{i=1}^{n} w_i \mathbf{I}[y_{i} = c]
$$

where the decision function $\mathbf{I}$ is defined as follows:
$$
     \mathbf{I}[y_i = c] = \begin{cases}
 1 & \text{if } y_i = c \\
 0 & \text{otherwise}
\end{cases}
$$

In [1]:
from pandas import read_csv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = KNeighborsClassifier(n_neighbors=100, metric="euclidean")
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 69.000000 %
Weighted F1-Score: 68.612504 %


### Regression
Averages the output of the neighbors to compute the output, also we can add a weight $w_i$ to give more significance to nearby points.

$$
    \hat{y}(x) = \sum_{i=1}^{n} w_i y_i
$$

In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = RadiusNeighborsRegressor(radius=199, metric="euclidean", weights="distance", n_jobs=-1)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 3499361006.672279
Mean absolute error: 16762.432838


/usr/local/lib/python3.13/dist-packages/numpy/_core/numeric.py:457: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


## Most common hyperparameters
- **`n_neighbors`:** Number of neighbors to use on KNN algorithm. Only available at `KNeighbors*` classes.
- **`radius`:** Range $r$ of parameter space. Only available at `RadiusNeighbors*` classes.
- **`weights`:** Function to weight training samples.
- **`algorithm`:** Algorithm used to compute nearest neighbors.
- **`metric`:** Metric to use for distance computation.
- **`leaf_size`:** Leaf size passed to ball tree or KD tree.
- **`p`:** Power parameter for the Minkowski metric.

## Advantages
- Generally quick at training.
- Performs well when the dataset has a clearly defined decision boundary.
- Sample weighting can improve model accuracy.

## Disadvantages
- Poor generalization capabilities.
- Slow at inference.
- Requires data standarization.
- Sensitivity to outliners.